# SparkCity Weather Feature — Day 1

**Owner:** Matthew Baise  
**Dataset:** `weather_data.parquet`

## Purpose

This notebook loads and explores SparkCity weather-station data using PySpark. It verifies the schema, measures data quality, summarizes weather readings, and prepares timestamp-derived features for later analysis.

## 1. Schema and Sample Records

The weather dataset should contain station identity, timestamp, geographic location, and six weather measurements.

In [35]:
expected_columns = [
    "station_id",
    "timestamp",
    "location_lat",
    "location_lon",
    "temperature",
    "humidity",
    "wind_speed",
    "wind_direction",
    "precipitation",
    "pressure",
]

weather_df.printSchema()
weather_df.show(5, truncate=False)

missing_columns = [
    column for column in expected_columns
    if column not in weather_df.columns
]

unexpected_columns = [
    column for column in weather_df.columns
    if column not in expected_columns
]

print(f"Missing columns: {missing_columns}")
print(f"Unexpected columns: {unexpected_columns}")

root
 |-- station_id: string (nullable = true)
 |-- timestamp: timestamp_ntz (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- pressure: double (nullable = true)

+----------+-------------------+------------+------------+-----------+--------+----------+--------------+-------------+--------+
|station_id|timestamp          |location_lat|location_lon|temperature|humidity|wind_speed|wind_direction|precipitation|pressure|
+----------+-------------------+------------+------------+-----------+--------+----------+--------------+-------------+--------+
|WTH-0001  |2025-01-01 00:00:00|40.680459   |-74.049119  |51.69      |62.77   |22.56     |287.0         |0.371        |1004.5  |
|WTH-0002  |2025-01-01 00:30:00|40.680695   

## 2. Dataset Coverage

This section measures the number of readings, number of stations, and timestamp range represented in the weather data.

In [36]:
coverage_df = weather_df.agg(
    F.count("*").alias("record_count"),
    F.countDistinct("station_id").alias("station_count"),
    F.min("timestamp").alias("first_timestamp"),
    F.max("timestamp").alias("last_timestamp"),
)

coverage_df.show(truncate=False)

+------------+-------------+-------------------+-------------------+
|record_count|station_count|first_timestamp    |last_timestamp     |
+------------+-------------+-------------------+-------------------+
|36000       |1200         |2025-01-01 00:00:00|2027-01-20 23:30:00|
+------------+-------------+-------------------+-------------------+



In [37]:
weather_report = validate_dataframe(weather_df, "weather")

print(f"Valid dataset: {weather_report['valid']}")
print(f"Record count: {weather_report['record_count']:,}")
print(f"Missing columns: {weather_report['missing_columns']}")
print(f"Non-numeric columns: {weather_report['non_numeric_columns']}")
print(f"Null counts: {weather_report['null_counts']}")
print(f"Duplicate count: {weather_report['duplicate_count']}")
print(f"Range violations: {weather_report['range_violations']}")
print(f"Timestamp violations: {weather_report['timestamp_violations']}")

Valid dataset: True
Record count: 36,000
Missing columns: []
Non-numeric columns: []
Null counts: {}
Duplicate count: 0
Range violations: {}
Timestamp violations: {}


In [38]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

from sparkcityx.data_quality import validate_dataframe
from sparkcityx.loaders import load_dataset

In [39]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SparkCity-Weather-Day1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")

Spark version: 4.2.0
Spark master: local[*]


In [40]:
weather_path = Path("../data/raw/weather_data.parquet")

if not weather_path.exists():
    weather_path = Path("data/raw/weather_data.parquet")

weather_df = load_dataset(spark, weather_path)

print(f"Source: {weather_path.resolve()}")
print(f"Records: {weather_df.count():,}")
print(f"Columns: {len(weather_df.columns)}")

Source: /Users/matthew/Projects/SparkCity_Capstone/data/raw/weather_data.parquet
Records: 36,000
Columns: 10


## 3. Weather Measurement Summary

The following statistics help identify unusual values and establish the normal range of each weather measurement.

In [41]:
measurement_columns = [
    "temperature",
    "humidity",
    "wind_speed",
    "wind_direction",
    "precipitation",
    "pressure",
]

weather_df.select(*measurement_columns).summary(
    "count",
    "mean",
    "stddev",
    "min",
    "max",
).show(truncate=False)

+-------+------------------+-----------------+------------------+------------------+-------------------+------------------+
|summary|temperature       |humidity         |wind_speed        |wind_direction    |precipitation      |pressure          |
+-------+------------------+-----------------+------------------+------------------+-------------------+------------------+
|count  |36000             |36000            |36000             |36000             |36000              |36000             |
|mean   |52.057066666666124|57.90195944444474|13.814496944444446|179.2382163888882 |0.11805058333333336|1012.0326983333297|
|stddev |11.425975557860184|8.952971188807014|4.962036676657104 |104.08542449107807|0.1303668096812493 |5.740256799888593 |
|min    |33.01             |41.07            |2.1               |0.02              |0.0                |1002.0            |
|max    |70.98             |74.98            |23.98             |360.0             |0.824              |1022.0            |
+-------

In [42]:
readings_per_station_df = (
    weather_df
    .groupBy("station_id")
    .agg(F.count("*").alias("reading_count"))
)

readings_per_station_df.summary(
    "count",
    "mean",
    "min",
    "max",
).show(truncate=False)

+-------+----------+-------------+
|summary|station_id|reading_count|
+-------+----------+-------------+
|count  |1200      |1200         |
|mean   |NULL      |30.0         |
|min    |WTH-0001  |30           |
|max    |WTH-1200  |30           |
+-------+----------+-------------+



## 4. Initial Timestamp Transformations

Derived time columns prepare the weather data for hourly, daily, weekly, and seasonal analysis.

In [43]:
weather_day1_df = (
    weather_df
    .withColumn("reading_date", F.to_date("timestamp"))
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withColumn("week_of_year", F.weekofyear("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
)

weather_day1_df.select(
    "station_id",
    "timestamp",
    "reading_date",
    "year",
    "month",
    "week_of_year",
    "day_of_week",
    "hour",
).show(10, truncate=False)

+----------+-------------------+------------+----+-----+------------+-----------+----+
|station_id|timestamp          |reading_date|year|month|week_of_year|day_of_week|hour|
+----------+-------------------+------------+----+-----+------------+-----------+----+
|WTH-0001  |2025-01-01 00:00:00|2025-01-01  |2025|1    |1           |4          |0   |
|WTH-0002  |2025-01-01 00:30:00|2025-01-01  |2025|1    |1           |4          |0   |
|WTH-0003  |2025-01-01 01:00:00|2025-01-01  |2025|1    |1           |4          |1   |
|WTH-0004  |2025-01-01 01:30:00|2025-01-01  |2025|1    |1           |4          |1   |
|WTH-0005  |2025-01-01 02:00:00|2025-01-01  |2025|1    |1           |4          |2   |
|WTH-0006  |2025-01-01 02:30:00|2025-01-01  |2025|1    |1           |4          |2   |
|WTH-0007  |2025-01-01 03:00:00|2025-01-01  |2025|1    |1           |4          |3   |
|WTH-0008  |2025-01-01 03:30:00|2025-01-01  |2025|1    |1           |4          |3   |
|WTH-0009  |2025-01-01 04:00:00|2025-01-01 

In [44]:
derived_columns = [
    "reading_date",
    "year",
    "month",
    "week_of_year",
    "day_of_week",
    "hour",
]

print(f"Original columns: {len(weather_df.columns)}")
print(f"Transformed columns: {len(weather_day1_df.columns)}")
print(
    "Derived columns present:",
    all(column in weather_day1_df.columns for column in derived_columns),
)

Original columns: 10
Transformed columns: 16
Derived columns present: True


## 5. Geographic Zone Mapping

Each weather station is mapped to a city zone when its latitude and longitude fall within that zone’s geographic boundaries.

In [45]:
zones_path = Path("../data/reference/city_zones.csv")

if not zones_path.exists():
    zones_path = Path("data/reference/city_zones.csv")

zones_df = load_dataset(spark, zones_path)

print(f"Zone records: {zones_df.count():,}")
zones_df.printSchema()
zones_df.show(5, truncate=False)

Zone records: 5
root
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- lat_min: double (nullable = true)
 |-- lat_max: double (nullable = true)
 |-- lon_min: double (nullable = true)
 |-- lon_max: double (nullable = true)
 |-- population: integer (nullable = true)

+--------+------------------+-----------+-------+-------+-------+-------+----------+
|zone_id |zone_name         |zone_type  |lat_min|lat_max|lon_min|lon_max|population|
+--------+------------------+-----------+-------+-------+-------+-------+----------+
|ZONE_001|Downtown Manhattan|commercial |40.71  |40.72  |-74.01 |-73.99 |250000    |
|ZONE_002|Midtown East      |commercial |40.725 |40.735 |-73.995|-73.975|180000    |
|ZONE_003|Upper East Side   |residential|40.745 |40.755 |-73.98 |-73.96 |220000    |
|ZONE_004|Astoria Queens    |residential|40.75  |40.76  |-73.99 |-73.97 |195000    |
|ZONE_005|Long Island City  |industrial |40.76  |40.77  |-73.98

In [46]:
weather_alias = weather_day1_df.alias("weather")
zone_alias = F.broadcast(zones_df).alias("zone")

zone_condition = (
    F.col("weather.location_lat").between(
        F.col("zone.lat_min"),
        F.col("zone.lat_max"),
    )
    & F.col("weather.location_lon").between(
        F.col("zone.lon_min"),
        F.col("zone.lon_max"),
    )
)

weather_zoned_df = (
    weather_alias
    .join(zone_alias, zone_condition, "left")
    .select(
        "weather.*",
        F.col("zone.zone_id").alias("zone_id"),
        F.col("zone.zone_name").alias("zone_name"),
        F.col("zone.zone_type").alias("zone_type"),
    )
)

In [47]:
mapping_summary_df = weather_zoned_df.agg(
    F.count("*").alias("joined_rows"),
    F.sum(
        F.when(F.col("zone_id").isNotNull(), 1).otherwise(0)
    ).alias("mapped_rows"),
    F.sum(
        F.when(F.col("zone_id").isNull(), 1).otherwise(0)
    ).alias("unmapped_rows"),
)

mapping_summary_df.show(truncate=False)

+-----------+-----------+-------------+
|joined_rows|mapped_rows|unmapped_rows|
+-----------+-----------+-------------+
|36060      |1277       |34783        |
+-----------+-----------+-------------+



In [48]:
duplicate_zone_mappings = (
    weather_zoned_df
    .groupBy("station_id", "timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate zone mappings: {duplicate_zone_mappings:,}")

Duplicate zone mappings: 60


In [49]:
(
    weather_zoned_df
    .groupBy("zone_id", "zone_name", "zone_type")
    .agg(
        F.count("*").alias("weather_readings"),
        F.countDistinct("station_id").alias("weather_stations"),
    )
    .orderBy(F.desc("weather_readings"))
    .show(truncate=False)
)

+--------+------------------+-----------+----------------+----------------+
|zone_id |zone_name         |zone_type  |weather_readings|weather_stations|
+--------+------------------+-----------+----------------+----------------+
|NULL    |NULL              |NULL       |34783           |1200            |
|ZONE_001|Downtown Manhattan|commercial |296             |65              |
|ZONE_003|Upper East Side   |residential|269             |62              |
|ZONE_004|Astoria Queens    |residential|239             |48              |
|ZONE_002|Midtown East      |commercial |238             |52              |
|ZONE_005|Long Island City  |industrial |235             |48              |
+--------+------------------+-----------+----------------+----------------+



## 6. Day 1 Data-Quality Findings

The shared SparkCity validator checks required columns, nulls, duplicate station/timestamp keys, numeric types, timestamps, geographic coordinates, humidity, wind, precipitation, and pressure.

The source does not formally declare the temperature unit. Therefore, the original temperature values are preserved without conversion until the team confirms whether they represent Fahrenheit or Celsius.

In [50]:
coverage = coverage_df.first().asDict()
mapping = mapping_summary_df.first().asDict()
station_summary = readings_per_station_df.agg(
    F.min("reading_count").alias("minimum"),
    F.max("reading_count").alias("maximum"),
    F.avg("reading_count").alias("average"),
).first().asDict()

print("DAY 1 WEATHER SUMMARY")
print("=" * 50)
print(f"Records: {coverage['record_count']:,}")
print(f"Stations: {coverage['station_count']:,}")
print(f"First timestamp: {coverage['first_timestamp']}")
print(f"Last timestamp: {coverage['last_timestamp']}")
print(f"Minimum readings per station: {station_summary['minimum']}")
print(f"Maximum readings per station: {station_summary['maximum']}")
print(f"Average readings per station: {station_summary['average']:.2f}")
print(f"Dataset valid: {weather_report['valid']}")
print(f"Duplicate readings: {weather_report['duplicate_count']}")
print(f"Mapped rows: {mapping['mapped_rows']:,}")
print(f"Unmapped rows: {mapping['unmapped_rows']:,}")
print(f"Duplicate zone mappings: {duplicate_zone_mappings:,}")

DAY 1 WEATHER SUMMARY
Records: 36,000
Stations: 1,200
First timestamp: 2025-01-01 00:00:00
Last timestamp: 2027-01-20 23:30:00
Minimum readings per station: 30
Maximum readings per station: 30
Average readings per station: 30.00
Dataset valid: True
Duplicate readings: 0
Mapped rows: 1,277
Unmapped rows: 34,783
Duplicate zone mappings: 60


In [51]:
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.
